# Stage 0 · GPU notebook

Bounded loader qualification, paired greedy collection, and fixed-context KL/effective writes. All inference uses frozen sources, the recorded runtime and the shared 20-unit budget. Model weights must already exist in the verified local Drive snapshot.


Ticket 10: set `CONDITIONAL_EXTENSION = True` in section 1 only after CPU sections 26–28 produce an applicable catalogue. The same qualification/collection/review cells then use separately named extension products. Keep the same cumulative Stage 0 usage ledger; the ceiling remains 20 units. Run sections 5–19 after setup (1–4), retaining every review/budget stop. The CPU notebook then uses the same flag for sections 17–22.


Primary batch16 path: reuse completed CPU products and diagnostic. Run 9B–9F, then12/12A and collection. Sections10–11 retain the singleton/conditional path only. No production starts before review of the committed plan summary.


## 1. Storage and checkout
Mount Drive when in Colab; select the existing Stage 0 release checkout. CPU runtime can prepare the plan; one recorded CUDA GPU is required only for the diagnostic run.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

RELEASE_COMMIT = 'd65e52d11d7e8aee737b33e33baa72033470cca6'
REPOSITORY_URL = 'https://github.com/mahadikprasad15/spar-pilot.git'
# Only the dedicated Colab checkout is pinned; local work is never switched.
REPO_DIR = Path('/content/spar-stage0') if Path('/content').is_dir() else Path.cwd()
ARTIFACT_ROOT = Path('/content/drive/MyDrive/SPAR/spar-pilot/artifacts') if Path('/content').is_dir() else REPO_DIR / 'artifacts'
STAGE0_NAME = 'stage0-geometry-v1'
CONDITIONAL_EXTENSION = False  # True selects the separately named Ticket 10 workflow.
USE_BATCH16 = not CONDITIONAL_EXTENSION  # Approved primary schedule: behavior16, internals1.
BATCH_NAME = STAGE0_NAME + '-behavior-b16-v2'  # v1 failed the superseded 3E batch gate; evidence preserved (ADR 0014).
PLAN = ARTIFACT_ROOT / 'plans' / STAGE0_NAME / 'stage0.plan.json'

if Path('/content').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
if Path('/content').is_dir():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPOSITORY_URL, str(REPO_DIR)], check=True)
    changed = subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True)
    if changed.strip():
        raise RuntimeError('The dedicated checkout has local changes. Preserve them before selecting a release.')
    subprocess.run(['git', 'fetch', 'origin', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', RELEASE_COMMIT], cwd=REPO_DIR, check=True)
if not (REPO_DIR / 'pilot_eval').is_dir():
    raise RuntimeError('Select the repository checkout before running the workflow.')

def run_stage0(operation, *arguments):
    """Stream the command's output; on failure, raise with its last lines so the real error is visible."""
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation,
               '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    try:
        for line in process.stdout:
            print(line, end='')
            tail = (tail + [line])[-30:]
        process.wait()
    except BaseException:
        # Stop the worker too: Ctrl-C first so it records timing and releases its writer,
        # then terminate, then kill; always reap it so no GPU job outlives the cell.
        import signal
        process.send_signal(signal.SIGINT)
        for escalate, timeout in [(process.terminate, 60), (process.kill, 30), (None, None)]:
            try:
                process.wait(timeout=timeout)
                break
            except subprocess.TimeoutExpired:
                escalate()
        raise
    if process.returncode:
        raise RuntimeError(f'stage0-{operation} failed (exit {process.returncode}):\n' + ''.join(tail))
    return subprocess.CompletedProcess(command, 0)


## 2. Select the pinned dependency environment
Run once in a fresh Colab runtime. Set the install option only when the pinned packages are missing or the next import check fails. Installation streams its output, does not load model weights, and removes the optional vision/audio packages that conflict with this text-only environment. Keep the kernel free of model-library imports while changing these packages.


In [ ]:
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from pilot_eval.sft_backend import PINS

INSTALL_GPU_DEPENDENCIES = False
if INSTALL_GPU_DEPENDENCIES:
    subprocess.run([sys.executable, '-u', '-m', 'pip', 'install',
                    *[f'{package}=={version}' for package, version in PINS.items()]],
                   cwd=REPO_DIR, check=True)
    subprocess.run([sys.executable, '-u', '-m', 'pip', 'uninstall', '-y',
                    'torchvision', 'torchaudio'], cwd=REPO_DIR, check=True)
print('Required package pins:', PINS)


## 3. Check actual text-model imports
Checks the pinned versions and actual Qwen/PEFT/TRL imports in a fresh child process. It records no scientific qualification and loads no model. CPU preparation may continue without CUDA; the later runtime check requires exactly one visible GPU.


In [ ]:
from pilot_eval.notebook_session import NotebookSession

environment_check = NotebookSession(ARTIFACT_ROOT, REPO_DIR, plan_name=STAGE0_NAME)
ENVIRONMENT = environment_check.check_environment(require_gpu=False)


## 4. Restore source readiness and budget status
Run after setup in a fresh kernel. Unknown prior Colab usage is not treated as zero.


In [ ]:
run_stage0('status', '--config', PLAN)


## 5. Prepare the qualification plan (CPU safe)
This binds the completed Ticket 03 adapter catalogue to the original ordered Pilot 4 150-question evaluation config. It does not load a model or use the GPU.

In [ ]:
EDIT_NAME = STAGE0_NAME + '-principal-edits-v1'
EDIT_PLAN = ARTIFACT_ROOT / 'plans' / EDIT_NAME / 'stage0.edits.json'
PILOT4_EVALUATION_CONFIG = ARTIFACT_ROOT / 'reports' / 'pilot4-results-so-far-2026-10-13' / 'inputs' / 'evaluation-config.json'
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
QUALIFICATION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.qualification.json'
assert EDIT_PLAN.is_file(), 'Complete CPU notebook sections 10–12 and restore the edit catalogue first.'
assert PILOT4_EVALUATION_CONFIG.is_file(), 'Restore the original Pilot 4 evaluation-config.json under artifacts/reports/.../inputs/.'
if CONDITIONAL_EXTENSION:
    CONDITIONAL_PLAN = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-intruder-edits-v1') / 'stage0.intruder-edits.json'
    assert CONDITIONAL_PLAN.is_file(), 'Complete CPU sections 26–28 first; do not allocate GPU time for a not-applicable catalogue.'
    run_stage0('qualification-prepare', '--catalogue', CONDITIONAL_PLAN, '--cohort', PILOT4_EVALUATION_CONFIG, '--name', QUALIFICATION_NAME)
else:
    run_stage0('qualification-prepare', '--edits', EDIT_PLAN, '--cohort', PILOT4_EVALUATION_CONFIG, '--name', QUALIFICATION_NAME)
selected = json.loads(QUALIFICATION_PLAN.read_text())
print('Qualification units:', selected['total_units'], '| production variants:', list(selected['variants']))


## 6. Check this GPU session and Drive readback
This cell checks the pinned runtime, visible single GPU, stable session ID, budget ledger, and a small write/read/delete on Drive. It does not load the model or spend inference time.

In [ ]:
def run_stage0_json(operation, *arguments):
    command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-' + operation, '--output-root', str(ARTIFACT_ROOT), *map(str, arguments)]
    completed = subprocess.run(command, cwd=REPO_DIR, check=True, capture_output=True, text=True)
    value = json.loads(completed.stdout)
    print(json.dumps(value, indent=2))
    return value

PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
QUALIFICATION_SESSION_ID = PREFLIGHT['runtime']['session_id']
assert PREFLIGHT['storage']['passed']
print('Prior cumulative usage:', PREFLIGHT['budget'])
if PREFLIGHT['budget']['spent_units'] is not None:
    assert PREFLIGHT['budget']['spent_units'] < 20, 'The cumulative Stage 0 ceiling is reached.'
assert not PREFLIGHT['active_reservations'], 'Reconcile the earlier reservation before starting another GPU job.'


## 7. Record the current cumulative Stage 0 usage (CPU safe)
Read the current Colab compute-unit total and enter it below. This is a cumulative Stage 0 reading across sessions, not the account balance. If you cannot establish prior usage, stop here and recover that number before reserving work.

In [ ]:
from datetime import datetime, timezone
import uuid

OBSERVED_STAGE0_UNITS = None  # Enter the actual cumulative Stage 0 compute-unit reading.
OBSERVATION_EVENT_ID = 'ticket04-start-' + uuid.uuid4().hex
if OBSERVED_STAGE0_UNITS is None:
    raise ValueError('Enter the observed cumulative Stage 0 usage; unknown is not zero.')
observation = {
    'event_id': OBSERVATION_EVENT_ID,
    'session_id': QUALIFICATION_SESSION_ID,
    'total_units': float(OBSERVED_STAGE0_UNITS),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-recorded cumulative Stage 0 Colab compute units before Ticket 04.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'ticket04-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
assert PREFLIGHT['budget']['state'] == 'observed' and PREFLIGHT['budget']['spent_units'] < 20


## 8. Approve and run the bounded diagnostic
Review the session and remaining budget from section 7. Set a conservative Colab unit rate and a wall-time cap for this diagnostic. The reservation is calculated before model load; interrupted work stays reserved until a newer cumulative usage reading is recorded.

Each of the 32 units compares an adapter reference with the mixed merged loader, the intentional double-application negative control, and greedy outputs. The run saves each completed unit and resumes only missing units.

In [ ]:
DIAGNOSTIC_REVIEWER = 'Prasad'
DIAGNOSTIC_NOTES = 'Bounded loader qualification; reviewed current Stage 0 usage and the selected 16-sequences-per-distinct-merged-layout scope.'
DIAGNOSTIC_SOURCE_REVIEWED = False  # Set True only after confirming the exact edits and cohort paths above.
DIAGNOSTIC_MAX_UNITS_PER_HOUR = None  # Enter a conservative rate based on the current Colab runtime.
DIAGNOSTIC_MAX_SECONDS = None  # Enter a hard wall-time cap for this diagnostic attempt.
if not DIAGNOSTIC_SOURCE_REVIEWED or DIAGNOSTIC_MAX_UNITS_PER_HOUR is None or DIAGNOSTIC_MAX_SECONDS is None:
    raise ValueError('Review the sources and enter conservative unit-rate and wall-time bounds before reserving GPU work.')
DIAGNOSTIC_APPROVAL = {
    'reviewer': DIAGNOSTIC_REVIEWER, 'notes': DIAGNOSTIC_NOTES,
    'session_id': QUALIFICATION_SESSION_ID,
    'max_units_per_hour': float(DIAGNOSTIC_MAX_UNITS_PER_HOUR),
    'max_seconds': float(DIAGNOSTIC_MAX_SECONDS),
    'source_reviewed': DIAGNOSTIC_SOURCE_REVIEWED
}
DIAGNOSTIC_APPROVAL_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'ticket04-diagnostic-approval.json'
DIAGNOSTIC_APPROVAL_PATH.write_text(json.dumps(DIAGNOSTIC_APPROVAL, indent=2) + '\n')
run_stage0('qualification-run', '--config', QUALIFICATION_PLAN, '--approval', DIAGNOSTIC_APPROVAL_PATH)


## 9. Verify and inspect calibration evidence
Verification is CPU-only. It checks all saved diagnostic units, token/mask identities, runtime binding, negative-control sensitivity and the hard stop. It does not itself approve the proposed 3E numerical allowance.

In [ ]:
run_stage0('qualification-verify', '--config', QUALIFICATION_PLAN)
DIAGNOSTIC_RESULT_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'results' / 'diagnostic.json'
DIAGNOSTIC_RESULT = json.loads(DIAGNOSTIC_RESULT_PATH.read_text())
print(json.dumps({key: DIAGNOSTIC_RESULT[key] for key in ['max_absolute_error', 'hard_stop', 'negative_control', 'greedy_acceptable', 'eligible_for_review', 'proposed_absolute_gate', 'production_estimate']}, indent=2))


## 9A. Reconcile diagnostic compute usage before resuming or freezing
After the diagnostic completes or stops, record the updated cumulative Stage 0 usage. This releases the diagnostic reservation while preserving the run's measured delta. You can record the reading from a later session after an interruption; the ledger records both run-session and observation-session IDs.

If the diagnostic is incomplete and the same Colab runtime is still connected, reconcile here before rerunning section 8. Sealed units resume only within that recorded runtime/session. If Colab restarted or the GPU/runtime changed, preserve this attempt and prepare a new named qualification; do not mix partial results across runtimes.

In [ ]:
CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC = None  # Enter the actual current cumulative Stage 0 unit reading.
if CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC is None:
    raise ValueError('Enter the observed cumulative usage before releasing the reservation.')
STATUS_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'meta' / 'status.json'
DIAGNOSTIC_RESULT_PATH = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'results' / 'diagnostic.json'
ATTEMPT_STATUS = json.loads(STATUS_PATH.read_text())
DIAGNOSTIC_RESULT = json.loads(DIAGNOSTIC_RESULT_PATH.read_text()) if DIAGNOSTIC_RESULT_PATH.exists() else {}
RESERVATION_ID = DIAGNOSTIC_RESULT.get('reservation_id') or ATTEMPT_STATUS.get('reservation_id')
assert RESERVATION_ID, 'No diagnostic reservation found to reconcile.'
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'ticket04-usage-observation.json'
observation = {
    'event_id': 'ticket04-after-diagnostic-' + uuid.uuid4().hex,
    'session_id': __import__('platform').node() + ':' + __import__('pathlib').Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_DIAGNOSTIC),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-recorded cumulative Stage 0 compute units after Ticket 04 diagnostic attempt.'
}
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', RESERVATION_ID)
PREFLIGHT = run_stage0_json('qualification-runtime', '--config', QUALIFICATION_PLAN)
assert PREFLIGHT['budget']['spent_units'] < 20, 'No unapproved work remains under the Stage 0 ceiling.'


## 9B. Prepare behavior batch agreement (CPU)
Primary path only. Reuses the completed numerical diagnostic, assembled adapters and successful capacity profile. No SVD/random-basis scan or model load. Check set: all10 variants × fixed16 and tail6 teacher-forced questions; V1/V5 greedy replay at both sizes (44 generated answers). Left padding. Exact token IDs and stop reasons pass; common-prefix divergences save position, top-two gap and measured error. Only measured near ties may pass; larger-gap or stop-metadata differences stop. Internals remain singleton and are not part of this check.


In [ ]:
assert USE_BATCH16 and not CONDITIONAL_EXTENSION
QUALIFICATION_NAME = STAGE0_NAME + '-loader-qualification-v1'
QUALIFICATION_PLAN = ARTIFACT_ROOT / 'plans' / QUALIFICATION_NAME / 'stage0.qualification.json'
DIAGNOSTIC_RESULT = json.loads((ARTIFACT_ROOT / 'runs/stage0' / QUALIFICATION_NAME / 'qualification/results/diagnostic.json').read_text())
BATCH_PLAN = ARTIFACT_ROOT / 'plans' / BATCH_NAME / 'stage0.batch.json'
BATCH_PROFILE = ARTIFACT_ROOT / 'runs/stage0' / (STAGE0_NAME + '-batch16-profile-v1') / 'batch-profile'
run_stage0('batch-prepare', '--qualification', QUALIFICATION_PLAN, '--profile', BATCH_PROFILE,
           '--name', BATCH_NAME, '--batch-size', 16)
BATCH_DIR = ARTIFACT_ROOT / json.loads(BATCH_PLAN.read_text())['run_path']
print('Batch plan:', BATCH_PLAN)
print('Qualification units:', len(json.loads(BATCH_PLAN.read_text())['units']))


## 9C. Run or resume the bounded behavior agreement (GPU)
Enter current cumulative Stage0 usage. Reconcile earlier active profiles before reserving this workload. This check does not authorize production; failures preserve evidence and stop. The same 20-unit ceiling applies.


In [ ]:
BATCH_SPENT_BEFORE = None
BATCH_CHECK_REVIEWED = False
BATCH_CHECK_SECONDS = 600.0
BATCH_CHECK_RATE = 1.6
if BATCH_SPENT_BEFORE is None or not BATCH_CHECK_REVIEWED:
    raise ValueError('Enter actual cumulative units and review the bounded check set above.')
import uuid
from datetime import datetime, timezone
from pilot_eval.stage0_budget import reservations
observation = BATCH_DIR / ('usage-before-' + uuid.uuid4().hex + '.json')
observation.write_text(json.dumps(dict(event_id=observation.stem,
    session_id=DIAGNOSTIC_RESULT['runtime']['session_id'], total_units=float(BATCH_SPENT_BEFORE),
    observed_at=datetime.now(timezone.utc).isoformat(), note='Actual cumulative usage before behavior batch agreement.')))
run_stage0('budget-record', '--config', PLAN, '--observation', observation)
for rid, row in reservations(ARTIFACT_ROOT).items():
    if row['state'] == 'active': run_stage0('budget-settle', '--reservation-id', rid)
BATCH_CHECK_APPROVAL = BATCH_DIR / 'budget-review.json'
BATCH_CHECK_APPROVAL.write_text(json.dumps(dict(reviewer='Prasad',
    notes='Reviewed behavior-only full16/tail6 agreement; no production authorization.',
    session_id=DIAGNOSTIC_RESULT['runtime']['session_id'],source_reviewed=True,
    max_units_per_hour=BATCH_CHECK_RATE,max_seconds=BATCH_CHECK_SECONDS)))
run_stage0('batch-run', '--config', BATCH_PLAN, '--approval', BATCH_CHECK_APPROVAL)


## 9D. Inspect evidence and save the numerical/batch review (CPU)
Review exact disagreements, stop metadata, measured near-tie count, integrity and forecast. Batched-vs-singleton FP32 logits are judged against the pre-registered 1e-3 hard stop (ADR 0014); the loader's evidence-derived 3E stays unchanged and is recorded alongside. Approval is separate from the production-budget freeze.


In [ ]:
run_stage0('batch-verify', '--config', BATCH_PLAN)
BATCH_REPORT = json.loads((BATCH_DIR / 'results/agreement.json').read_text())
print(json.dumps(BATCH_REPORT, indent=2))
BATCH_NUMERICAL_REVIEWED = False
BATCH_NUMERICAL_NOTES = ''
if not BATCH_NUMERICAL_REVIEWED or not BATCH_NUMERICAL_NOTES.strip():
    raise ValueError('Review the displayed evidence before saving approval.')
import hashlib
BATCH_NUMERICAL_REVIEW = BATCH_DIR / 'numerical-review.json'
BATCH_NUMERICAL_REVIEW.write_text(json.dumps(dict(reviewer='Prasad',notes=BATCH_NUMERICAL_NOTES,
    agreement_sha256=hashlib.sha256((BATCH_DIR/'complete.json').read_bytes()).hexdigest(),
    absolute_gate=BATCH_REPORT['absolute_gate'],batch_gate=BATCH_REPORT['batch_gate'])))
run_stage0('batch-approve', '--config', BATCH_PLAN, '--review', BATCH_NUMERICAL_REVIEW)


## 9E. Preview the production plan and reconcile agreement usage (CPU)
Send this concrete summary for approval before production. The committed code hash is recorded. The forecast is generation-only; internals use a separate singleton budget. This cell does not launch production.


In [ ]:
BATCH_SPENT_AFTER = None
if BATCH_SPENT_AFTER is None: raise ValueError('Enter actual cumulative usage after agreement.')
observation = BATCH_DIR / ('usage-after-' + uuid.uuid4().hex + '.json')
observation.write_text(json.dumps(dict(event_id=observation.stem,
    session_id=BATCH_REPORT['runtime']['session_id'], total_units=float(BATCH_SPENT_AFTER),
    observed_at=datetime.now(timezone.utc).isoformat(), note='Actual cumulative usage after behavior agreement.')))
run_stage0('budget-record', '--config', PLAN, '--observation', observation)
for rid, row in reservations(ARTIFACT_ROOT).items():
    if row['state'] == 'active': run_stage0('budget-settle', '--reservation-id', rid)
BATCH_PRODUCTION_RATE = 1.6
BATCH_PRODUCTION_SECONDS = BATCH_REPORT['forecast']['conservative_total_seconds']
BATCH_PLAN_SUMMARY = dict(code_git_commit=BATCH_REPORT['runtime']['git_commit'],
    variants=list(json.loads(QUALIFICATION_PLAN.read_text())['variants']),questions=150,
    behaviour_batch=16,tail_batch=6,behaviour_responses=1500,internals_batch=1,
    internals_sequences=3000,context_chunk=32,decoder_precision='float32',kl_head_precision='float64',
    decoding=json.loads(QUALIFICATION_PLAN.read_text())['decoding'],
    forecast=BATCH_REPORT['forecast'],max_units_per_hour=BATCH_PRODUCTION_RATE,
    behaviour_reservation_units=BATCH_PRODUCTION_RATE*BATCH_PRODUCTION_SECONDS/3600,
    current_cumulative_units=BATCH_SPENT_AFTER,ceiling_units=20)
(BATCH_DIR/'production-proposal.json').write_text(json.dumps(BATCH_PLAN_SUMMARY,indent=2))
print(json.dumps(BATCH_PLAN_SUMMARY,indent=2))


## 9F. Freeze the approved production plan (CPU)
Only after the displayed proposal is approved. Requires the same clean committed code revision as agreement preparation/run. The runner remains resumable and the reservation is bounded.


In [ ]:
BATCH_PRODUCTION_APPROVED = False
BATCH_PRODUCTION_NOTES = ''
if not BATCH_PRODUCTION_APPROVED or not BATCH_PRODUCTION_NOTES.strip():
    raise ValueError('Obtain approval of the concrete plan summary before freezing production.')
BATCH_PRODUCTION_REVIEW = BATCH_DIR/'production-review.json'
BATCH_PRODUCTION_REVIEW.write_text(json.dumps(dict(reviewer='Prasad',notes=BATCH_PRODUCTION_NOTES,
    agreement_sha256=hashlib.sha256((BATCH_DIR/'complete.json').read_bytes()).hexdigest(),
    max_units_per_hour=BATCH_PRODUCTION_RATE,max_seconds=BATCH_PRODUCTION_SECONDS,
    session_id=BATCH_REPORT['runtime']['session_id'],source_reviewed=True)))
run_stage0('batch-freeze','--config',BATCH_PLAN,'--review',BATCH_PRODUCTION_REVIEW)
print('Continue with 12, 12A, then the existing collection cells. Internals remain at batch1.')


## 10. Save the measured numerical-gate review
Only approve after reviewing the saved maximum error, every answer-position maximum, the worst coordinate, negative-control ratios and any common-prefix near-tie exceptions. Enter the exact evidence-derived 3E allowance shown below. A diagnostic above 1e-3 or an insensitive negative control cannot be approved.

In [ ]:
if USE_BATCH16:
    print('Use batch16 reviews and freeze in 9B–9F; the singleton freeze is not used.')
else:
    NUMERICAL_REVIEWED = False  # Set True after inspecting the complete diagnostic JSON and saved unit records.
    NUMERICAL_REVIEWER = 'Prasad'
    NUMERICAL_REVIEW_NOTES = ''
    if not NUMERICAL_REVIEWED or not NUMERICAL_REVIEW_NOTES.strip() or not DIAGNOSTIC_RESULT['eligible_for_review']:
        raise ValueError('Review eligible diagnostic evidence and enter notes before approving the numerical allowance.')
    DIAGNOSTIC_COMPLETE = ARTIFACT_ROOT / 'runs' / 'stage0' / QUALIFICATION_NAME / 'qualification' / 'complete.json'
    NUMERICAL_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'ticket04-numerical-review.json'
    NUMERICAL_REVIEW = {
        'reviewer': NUMERICAL_REVIEWER, 'notes': NUMERICAL_REVIEW_NOTES,
        'diagnostic_sha256': __import__('hashlib').sha256(DIAGNOSTIC_COMPLETE.read_bytes()).hexdigest(),
        'absolute_gate': DIAGNOSTIC_RESULT['proposed_absolute_gate']
    }
    NUMERICAL_REVIEW_PATH.write_text(json.dumps(NUMERICAL_REVIEW, indent=2) + '\n')
    run_stage0('qualification-approve', '--config', QUALIFICATION_PLAN, '--review', NUMERICAL_REVIEW_PATH)


## 11. Approve the measured production reservation and freeze
Enter a conservative runtime unit rate and wall-time allowance that covers the measured production forecast. The frozen plan covers the ten-variant, 150-question paired greedy evaluation; later KL/effective-write jobs need their own measured reservations under the same 20-unit ceiling.

In [ ]:
if USE_BATCH16:
    print('Use batch16 reviews and freeze in 9B–9F; the singleton freeze is not used.')
else:
    PRODUCTION_REVIEWED = False  # Set True after reviewing the full measured forecast.
    PRODUCTION_REVIEWER = 'Prasad'
    PRODUCTION_REVIEW_NOTES = ''
    PRODUCTION_MAX_UNITS_PER_HOUR = None  # Enter a conservative rate from Colab's current compute-unit display.
    PRODUCTION_MAX_SECONDS = None  # Must cover conservative_total_seconds below.
    if (not PRODUCTION_REVIEWED or not PRODUCTION_REVIEW_NOTES.strip()
            or PRODUCTION_MAX_UNITS_PER_HOUR is None or PRODUCTION_MAX_SECONDS is None):
        raise ValueError('Review the forecast and enter explicit unit/time bounds before freezing production work.')
    assert PRODUCTION_MAX_SECONDS >= DIAGNOSTIC_RESULT['production_estimate']['conservative_total_seconds']
    PRODUCTION_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'ticket04-production-budget-review.json'
    PRODUCTION_REVIEW = {
        'reviewer': PRODUCTION_REVIEWER, 'notes': PRODUCTION_REVIEW_NOTES,
        'diagnostic_sha256': __import__('hashlib').sha256(DIAGNOSTIC_COMPLETE.read_bytes()).hexdigest(),
        'max_units_per_hour': float(PRODUCTION_MAX_UNITS_PER_HOUR),
        'max_seconds': float(PRODUCTION_MAX_SECONDS),
        'session_id': DIAGNOSTIC_RESULT['runtime']['session_id'],
        'source_reviewed': True
    }
    PRODUCTION_REVIEW_PATH.write_text(json.dumps(PRODUCTION_REVIEW, indent=2) + '\n')
    run_stage0('qualification-freeze', '--config', QUALIFICATION_PLAN, '--review', PRODUCTION_REVIEW_PATH)


## 12. Restore and prepare paired greedy collection (CPU safe)
Consumes the frozen Ticket 04 execution. The approved primary path uses batch16 (tail6), one batch per durable unit. The legacy/conditional path uses singleton inference; eight sequential calls form one durable unit. The full catalogue is 10 variants × 150 questions. This cell displays the frozen work count and does not load a model. After a kernel restart, run section 1 and this section to restore the saved paths.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / (BATCH_NAME if USE_BATCH16 else QUALIFICATION_NAME) / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
INFERENCE_BATCH_SIZE = 16 if USE_BATCH16 else 1
BATCHES_PER_DURABLE_UNIT = 1 if USE_BATCH16 else 8
assert EXECUTION_PLAN.is_file(), 'Complete the numerical and production-budget reviews and freeze in sections 10–11 first.'
run_stage0('evaluation-prepare', '--execution', EXECUTION_PLAN,
           '--batch-size', INFERENCE_BATCH_SIZE, '--batches-per-unit', BATCHES_PER_DURABLE_UNIT)
EVALUATION_CONFIG = json.loads(EVALUATION_PLAN.read_text())
EVALUATION_DIR = ARTIFACT_ROOT / EVALUATION_CONFIG['run_path']
print('Variants:', EVALUATION_CONFIG['variant_order'])
print('Questions:', len(EVALUATION_CONFIG['item_ids']))
print('Model generation calls:', sum(len(unit['generation_batches']) for unit in EVALUATION_CONFIG['units']))
print('Durable units:', len(EVALUATION_CONFIG['units']))
print('Raw results:', EVALUATION_DIR / 'results')


## 12A. Freeze the primary comparison plan before collection (CPU safe)
Freezes the full comparison family, hypotheses, bootstrap procedure and −5 pp preservation rule before new results. Preparing the drift plan loads no model. If collection already started, the analysis is marked exploratory. The statistical NumPy runtime is recorded when the CPU analysis actually runs, independently of this GPU preparation environment.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / (BATCH_NAME if USE_BATCH16 else QUALIFICATION_NAME) / 'stage0.execution.json'
EVALUATION_PLAN = EXECUTION_PLAN.parent / 'stage0.evaluation.json'
DRIFT_NAME = STAGE0_NAME + ('-intruder-fixed-context-v1' if CONDITIONAL_EXTENSION else ('-batch16-fp64-fixed-context-v3' if USE_BATCH16 else '-fixed-context-v1'))
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
FINDINGS_NAME = STAGE0_NAME + ('-intruder-findings-v1' if CONDITIONAL_EXTENSION else ('-batch16-primary-findings-v3' if USE_BATCH16 else '-primary-findings-v1'))
FINDINGS_PLAN = ARTIFACT_ROOT / 'plans' / FINDINGS_NAME / 'stage0.findings.json'
FIXED_INPUTS = ARTIFACT_ROOT / 'plans' / 'pilot3-inputs-v1' / 'activation.prepared.json'
run_stage0('drift-prepare', '--execution', EXECUTION_PLAN, '--inputs', FIXED_INPUTS,
           '--name', DRIFT_NAME, '--batch-size', 1, '--batches-per-unit', 8,
           '--context-chunk', 32, '--workspace-bytes', 512 * 1024 * 1024)
# ADR 0015: the v2 drift identity inherits the unchanged v1 pre-collection registration.
REGISTERED_FROM = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-batch16-primary-findings-v1') / 'stage0.findings.json'
run_stage0('findings-prepare', '--evaluation', EVALUATION_PLAN, '--drift', DRIFT_PLAN, '--name', FINDINGS_NAME,
           *(['--registered-from', REGISTERED_FROM] if USE_BATCH16 and REGISTERED_FROM.is_file() else []))
registration = json.loads(FINDINGS_PLAN.read_text())
print('Registration:', registration['registration'])
print('Comparisons:', [row['id'] for row in registration['contrasts']])


## 13. Collect or resume paired greedy responses (GPU)
Loads one local FP32 model and the ten verified adapters in the recorded qualified runtime. It re-evaluates the SFT and GRPO references alongside every edit, then seals responses, token IDs, EOS/cap metadata, strict/flexible-v3 scores and provenance. Progress distinguishes generated responses pending a seal from durable completed units.

The initial run consumes section 11's active reservation. A normal same-session resume can use its remaining recorded wall time. After reconciling a stopped or exhausted reservation in section 14, explicitly approve a new bounded attempt using the settings below. A different runtime/session is rejected before loading the model.


In [ ]:
RENEW_RESERVATION = False
RESUME_REVIEWER = 'Prasad'
RESUME_REVIEW_NOTES = ''
RESUME_MAX_UNITS_PER_HOUR = None
RESUME_MAX_SECONDS = None

arguments = ['--config', EVALUATION_PLAN]
if RENEW_RESERVATION:
    if (not RESUME_REVIEW_NOTES.strip() or RESUME_MAX_UNITS_PER_HOUR is None
            or RESUME_MAX_SECONDS is None):
        raise ValueError('Enter reviewed notes and conservative rate/time limits after reconciling the earlier reservation.')
    review = {
        'reviewer': RESUME_REVIEWER, 'notes': RESUME_REVIEW_NOTES,
        'session_id': EVALUATION_CONFIG['runtime']['session_id'], 'source_reviewed': True,
        'max_units_per_hour': float(RESUME_MAX_UNITS_PER_HOUR),
        'max_seconds': float(RESUME_MAX_SECONDS),
    }
    RESUME_REVIEW_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'paired-resume-budget-review.json'
    RESUME_REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
    RESUME_REVIEW_PATH.write_text(json.dumps(review, indent=2) + '\n')
    arguments += ['--approval', RESUME_REVIEW_PATH]
run_stage0('evaluation-run', *arguments)


## 14. Reconcile collection usage (CPU safe)
After collection completes or stops, enter the actual cumulative Stage 0 compute-unit usage to reconcile its reservation. This does not change responses or reset the total budget. After a lost process, inspect `evaluation-status` and the saved writer token before using the documented explicit writer-recovery command.


In [ ]:
from datetime import datetime, timezone
import platform
import uuid

CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION = None
if CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION is None:
    raise ValueError('Enter the actual cumulative Stage 0 usage before settling this reservation.')
allocation = json.loads((EVALUATION_DIR / 'meta' / 'active-reservation.json').read_text())
observation = {
    'event_id': 'paired-after-' + uuid.uuid4().hex,
    'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_COLLECTION),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-observed cumulative Stage 0 consumption after paired collection attempt.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'paired-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', allocation['reservation_id'])
run_stage0('evaluation-status', '--config', EVALUATION_PLAN)


## 15. Verify and inspect the raw result (CPU)
Reconstructs scores and summary counts from every sealed batch and checks the aggregate response JSONL. Run this on CPU in the second notebook after GPU collection and usage reconciliation. Paired intervals, human-answer overlays and accuracy-preservation claims belong to Ticket 07.


In [ ]:
run_stage0('evaluation-verify', '--config', EVALUATION_PLAN)
from IPython.display import Markdown, display
summary = json.loads((EVALUATION_DIR / 'results' / 'results.json').read_text())
print('Verified responses:', summary['response_count'])
print('Execution kind:', summary['execution_kind'])
display(Markdown((EVALUATION_DIR / 'results' / 'results.md').read_text()))
print('Raw JSONL:', EVALUATION_DIR / 'results' / 'responses.jsonl')
print('Structured summary:', EVALUATION_DIR / 'results' / 'results.json')


## 16. Prepare fixed-context KL and writes (CPU safe)
Uses the original 150 GSM8K gold and 150 FineWeb sequences and all ten primary variants. Three bounded instrument controls use four documents each. Inference remains singleton; saving eight documents together does not change the model batch. Section 15 can be deferred to the CPU notebook. Reconcile the greedy reservation before starting the next GPU job.


In [ ]:
QUALIFICATION_NAME = STAGE0_NAME + ('-intruder-qualification-v1' if CONDITIONAL_EXTENSION else '-loader-qualification-v1')
EXECUTION_PLAN = ARTIFACT_ROOT / 'plans' / (BATCH_NAME if USE_BATCH16 else QUALIFICATION_NAME) / 'stage0.execution.json'
FIXED_INPUTS = ARTIFACT_ROOT / 'plans' / 'pilot3-inputs-v1' / 'activation.prepared.json'
DRIFT_NAME = STAGE0_NAME + ('-intruder-fixed-context-v1' if CONDITIONAL_EXTENSION else ('-batch16-fp64-fixed-context-v3' if USE_BATCH16 else '-fixed-context-v1'))
DRIFT_PLAN = ARTIFACT_ROOT / 'plans' / DRIFT_NAME / 'stage0.drift.json'
DRIFT_CONTEXT_CHUNK = 32
DRIFT_WORKSPACE_BYTES = 512 * 1024 * 1024
run_stage0('drift-prepare', '--execution', EXECUTION_PLAN, '--inputs', FIXED_INPUTS,
           '--name', DRIFT_NAME, '--batch-size', 1, '--batches-per-unit', 8,
           '--context-chunk', DRIFT_CONTEXT_CHUNK, '--workspace-bytes', DRIFT_WORKSPACE_BYTES)
DRIFT_CONFIG = json.loads(DRIFT_PLAN.read_text())
DRIFT_DIR = ARTIFACT_ROOT / DRIFT_CONFIG['run_path']
print('Main variant/document combinations:', (len(DRIFT_CONFIG['variants']) - 2) * len(DRIFT_CONFIG['item_ids']))
print('Durable units:', len(DRIFT_CONFIG['units']))
print('Bounded controls:', DRIFT_CONFIG['controls'])
print('Saved results:', DRIFT_DIR / 'results')


## 17. Bound, measure or resume fixed contexts (GPU)
Approve a separate allocation after recording actual cumulative usage and settling the previous reservation. The initial limit of four units collects three controls and the first SFT group; inspect its timing before continuing. Set the unit limit to `None` to finish under the remaining approved time. A new allocation requires usage reconciliation; an existing active allocation resumes without a new approval. Full-vocabulary probabilities are reduced in bounded chunks and discarded.


In [ ]:
APPROVE_NEW_DRIFT_BUDGET = False
DRIFT_MAX_UNITS_PER_HOUR = None
DRIFT_MAX_SECONDS = None
DRIFT_REVIEW_NOTES = ''
DRIFT_MAX_NEW_UNITS = 4
DRIFT_APPROVAL = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'drift-budget-review.json'
arguments = ['--config', DRIFT_PLAN]
if APPROVE_NEW_DRIFT_BUDGET:
    if DRIFT_MAX_UNITS_PER_HOUR is None or DRIFT_MAX_SECONDS is None or not DRIFT_REVIEW_NOTES.strip():
        raise ValueError('Enter a conservative current unit rate, bounded seconds and review notes.')
    import platform
    approval = {
        'reviewer': 'Prasad', 'notes': DRIFT_REVIEW_NOTES,
        'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
        'max_units_per_hour': float(DRIFT_MAX_UNITS_PER_HOUR),
        'max_seconds': float(DRIFT_MAX_SECONDS), 'source_reviewed': True
    }
    DRIFT_APPROVAL.parent.mkdir(parents=True, exist_ok=True)
    DRIFT_APPROVAL.write_text(json.dumps(approval, indent=2) + '\n')
    arguments += ['--approval', DRIFT_APPROVAL]
if DRIFT_MAX_NEW_UNITS is not None:
    arguments += ['--max-new-units', DRIFT_MAX_NEW_UNITS]
run_stage0('drift-run', *arguments)
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 18. Reconcile fixed-context usage (CPU safe)
Record the actual cumulative Stage 0 usage after completion or a stopped attempt. Settling the allocation does not delete sealed units. If more work is needed, a later bounded allocation resumes those units in the same qualified runtime.


In [ ]:
from datetime import datetime, timezone
import platform
import uuid

CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT = None
if CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT is None:
    raise ValueError('Enter actual cumulative Stage 0 compute-unit usage.')
allocation = json.loads((DRIFT_DIR / 'meta' / 'active-reservation.json').read_text())
observation = {
    'event_id': 'drift-after-' + uuid.uuid4().hex,
    'session_id': platform.node() + ':' + Path('/proc/sys/kernel/random/boot_id').read_text().strip(),
    'total_units': float(CUMULATIVE_STAGE0_UNITS_AFTER_DRIFT),
    'observed_at': datetime.now(timezone.utc).isoformat(),
    'note': 'Operator-observed cumulative Stage 0 usage after fixed-context attempt.'
}
OBSERVATION_PATH = ARTIFACT_ROOT / 'notebook' / QUALIFICATION_NAME / 'drift-usage-observation.json'
OBSERVATION_PATH.parent.mkdir(parents=True, exist_ok=True)
OBSERVATION_PATH.write_text(json.dumps(observation, indent=2) + '\n')
run_stage0('budget-record', '--config', PLAN, '--observation', OBSERVATION_PATH)
run_stage0('budget-settle', '--reservation-id', allocation['reservation_id'])
run_stage0('drift-status', '--config', DRIFT_PLAN)


## 19. Verify fixed-context results (CPU)
Reconstruct the document and corpus/view summaries from sealed scalar KL arrays and module write sums. No model is loaded. Equal-document means and token-weighted means remain separate; uncertainty and joined behavioral claims belong to Ticket 07.


In [ ]:
run_stage0('drift-verify', '--config', DRIFT_PLAN)
result = json.loads((DRIFT_DIR / 'results' / 'results.json').read_text())
print('Measurement complete:', result['measurement_complete'])
print('Execution kind:', result['execution_kind'])
print('Corpus/view cells:', len(result['cells']))
print('Per-document raw sums:', DRIFT_DIR / 'results' / 'per-document.jsonl')
print('Per-position KL and mappings:', DRIFT_DIR / 'units')


## CPU handoff
After GPU sections 13–14 and 17–18 finish collection and reconcile usage, release the GPU. CPU notebook sections 17–22 produce the primary report, optional human-label overlay, and raw data/figure exports. Supporting experiments and the conditional intruder extension do not block these primary findings.


## 20. Restore optional R0 and inspect this runtime
Run after GPU setup 1–4 and CPU sections 29–30. No primary/conditional qualification is required. This preflight does not load a model. Recorded original GPU/software mismatches disable the historical join and select a fresh uniform512 timing/profile route. R0 cannot use another active reservation.


In [ ]:
from datetime import datetime, timezone
import uuid

R0_NAME = STAGE0_NAME + '-untuned-rollouts-v1'
R0_PLAN = ARTIFACT_ROOT / 'plans' / R0_NAME / 'stage0.rollouts.json'
assert R0_PLAN.is_file(), 'Complete CPU R0 preparation first.'
R0_DIR = ARTIFACT_ROOT / json.loads(R0_PLAN.read_text())['run_path']
R0_NOTEBOOK = ARTIFACT_ROOT / 'notebook' / R0_NAME
R0_NOTEBOOK.mkdir(parents=True, exist_ok=True)

def r0_usage(total, note):
    path = R0_NOTEBOOK / ('usage-' + uuid.uuid4().hex + '.json')
    value = dict(event_id=path.stem, session_id=R0_PREFLIGHT['runtime']['session_id'],
                 total_units=float(total), observed_at=datetime.now(timezone.utc).isoformat(), note=note)
    path.write_text(json.dumps(value, indent=2) + '\n')
    run_stage0('budget-record', '--config', PLAN, '--observation', path)

def r0_allocation(rate, seconds, notes):
    return dict(reviewer='Prasad', notes=notes, session_id=R0_PREFLIGHT['runtime']['session_id'],
                max_units_per_hour=float(rate), max_seconds=float(seconds), source_reviewed=True)

command = [sys.executable, '-u', '-m', 'pilot_eval', 'stage0-rollouts-runtime',
           '--config', str(R0_PLAN), '--output-root', str(ARTIFACT_ROOT)]
R0_PREFLIGHT = json.loads(subprocess.run(command, cwd=REPO_DIR, check=True, capture_output=True, text=True).stdout)
print(json.dumps(R0_PREFLIGHT, indent=2))


## 21. Bound and run the two-group R0 profile (GPU)
For the first attempt, approve a new allocation using the actual cumulative Stage0 unit reading, a conservative unit rate and wall-time cap. The two groups retain eight draws each. For an interruption in the same session with remaining reserved time, leave new approval disabled and resume. Unknown elapsed time or a settled/exhausted reservation needs reconciliation and a fresh bounded allocation.


In [ ]:
R0_APPROVE_NEW_PROFILE_BUDGET = False
R0_SOURCE_REVIEWED = False
R0_SPENT_BEFORE_PROFILE = None
R0_PROFILE_UNITS_PER_HOUR = None
R0_PROFILE_MAX_SECONDS = None
R0_PROFILE_NOTES = ''
R0_PROFILE_MAX_NEW_GROUPS = 2
arguments = ['--config', R0_PLAN]
if R0_APPROVE_NEW_PROFILE_BUDGET:
    if (not R0_SOURCE_REVIEWED or R0_SPENT_BEFORE_PROFILE is None
            or R0_PROFILE_UNITS_PER_HOUR is None or R0_PROFILE_MAX_SECONDS is None
            or not R0_PROFILE_NOTES.strip()):
        raise ValueError('Review the exact profile scope and enter cumulative usage, conservative rate, time and notes.')
    r0_usage(R0_SPENT_BEFORE_PROFILE, 'Cumulative Stage0 units before optional R0 profile')
    approval = R0_NOTEBOOK / ('profile-budget-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(r0_allocation(R0_PROFILE_UNITS_PER_HOUR, R0_PROFILE_MAX_SECONDS, R0_PROFILE_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if R0_PROFILE_MAX_NEW_GROUPS is not None:
    arguments += ['--max-new-units', R0_PROFILE_MAX_NEW_GROUPS]
run_stage0('rollouts-profile', *arguments)
run_stage0('rollouts-status', '--config', R0_PLAN)


## 22. Inspect profile and reconcile its usage
This also reconciles a failed or stopped profile attempt. Completion is required before production freeze. Exact token/text/stop replay permits reuse; any mismatch or missing seed/order selects fresh512. Reward-rate similarity is never a gate. The forecast extrapolates measured work to the cap and is not a completion guarantee.


In [ ]:
R0_SPENT_AFTER_PROFILE = None
profile_path = R0_DIR / 'profile' / 'results' / 'profile.json'
R0_PROFILE = json.loads(profile_path.read_text()) if profile_path.exists() and (R0_DIR / 'profile' / 'complete.json').exists() else {}
if R0_PROFILE:
    run_stage0('rollouts-profile-verify', '--config', R0_PLAN)
    print(json.dumps(R0_PROFILE, indent=2))
profile_status = json.loads((R0_DIR / 'profile' / 'meta' / 'status.json').read_text())
reservation = R0_PROFILE.get('reservation_id') or profile_status.get('reservation_id')
if R0_SPENT_AFTER_PROFILE is None or not reservation:
    raise ValueError('Enter actual cumulative usage after the attempt; a reservation must exist before reconciliation.')
r0_usage(R0_SPENT_AFTER_PROFILE, 'Cumulative Stage0 units after optional R0 profile attempt')
run_stage0('budget-settle', '--reservation-id', reservation)


## 23. Review the route and freeze production
Separate approval covers either the missing384 groups or a fresh512 cohort in this profiled session. It must cover the measured forecast while staying within the cumulative 20-unit ceiling. Exact same-seed timing groups are reused where they match the frozen production identities.


In [ ]:
R0_PRODUCTION_REVIEWED = False
R0_PRODUCTION_UNITS_PER_HOUR = None
R0_PRODUCTION_MAX_SECONDS = None
R0_PRODUCTION_NOTES = ''
R0_PROFILE = json.loads((R0_DIR / 'profile' / 'results' / 'profile.json').read_text())
if (not R0_PRODUCTION_REVIEWED or R0_PRODUCTION_UNITS_PER_HOUR is None
        or R0_PRODUCTION_MAX_SECONDS is None or not R0_PRODUCTION_NOTES.strip()):
    raise ValueError('Review the route, exact evidence and measured forecast; enter rate, bounded seconds and notes.')
review = r0_allocation(R0_PRODUCTION_UNITS_PER_HOUR, R0_PRODUCTION_MAX_SECONDS, R0_PRODUCTION_NOTES)
review.update(profile_sha256=__import__('hashlib').sha256((R0_DIR / 'profile' / 'complete.json').read_bytes()).hexdigest(),
              route=R0_PROFILE['route'])
review_path = R0_NOTEBOOK / 'production-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('rollouts-freeze', '--config', R0_PLAN, '--review', review_path)


## 24. Collect or resume R0 (GPU)
Start with one group to inspect actual throughput, then set the limit to `None` to finish. Each completed eight-draw group is sealed and skipped on resume. Renew a reservation only after reconciling the old one. A changed session needs a separately named, newly profiled cohort; it cannot silently finish an old paired source.


In [ ]:
R0_MAX_NEW_GROUPS = 1
R0_RENEW_COLLECTION_BUDGET = False
R0_SPENT_BEFORE_RENEWAL = None
R0_RENEW_UNITS_PER_HOUR = None
R0_RENEW_MAX_SECONDS = None
R0_RENEW_NOTES = ''
arguments = ['--config', R0_PLAN]
if R0_RENEW_COLLECTION_BUDGET:
    if (R0_SPENT_BEFORE_RENEWAL is None or R0_RENEW_UNITS_PER_HOUR is None
            or R0_RENEW_MAX_SECONDS is None or not R0_RENEW_NOTES.strip()):
        raise ValueError('Reconcile the old reservation, then enter actual usage and new conservative limits.')
    r0_usage(R0_SPENT_BEFORE_RENEWAL, 'Cumulative Stage0 units before R0 collection renewal')
    approval = R0_NOTEBOOK / ('collection-budget-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(r0_allocation(R0_RENEW_UNITS_PER_HOUR, R0_RENEW_MAX_SECONDS, R0_RENEW_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if R0_MAX_NEW_GROUPS is not None:
    arguments += ['--max-new-units', R0_MAX_NEW_GROUPS]
run_stage0('rollouts-run', *arguments)
run_stage0('rollouts-status', '--config', R0_PLAN)


## 25. Reconcile R0 collection and hand off to CPU
Record actual cumulative usage after completion or a stopped attempt. Once collection and reconciliation finish, release the GPU and use CPU section31 for full verification, raw tables, memberships and the report. R0 does not run NLL or gradient measurements.


In [ ]:
R0_SPENT_AFTER_COLLECTION = None
collection_status = json.loads((R0_DIR / 'collection' / 'meta' / 'status.json').read_text())
reservation = collection_status.get('reservation_id')
if not reservation:
    reservation = json.loads((R0_DIR / 'collection' / 'meta' / 'active-reservation.json').read_text())['reservation_id']
if R0_SPENT_AFTER_COLLECTION is None:
    raise ValueError('Enter the actual cumulative Stage0 unit reading after collection or interruption.')
r0_usage(R0_SPENT_AFTER_COLLECTION, 'Cumulative Stage0 units after optional R0 collection attempt')
run_stage0('budget-settle', '--reservation-id', reservation)
if (R0_DIR / 'complete.json').exists():
    run_stage0('rollouts-verify', '--config', R0_PLAN)
    print('CPU handoff:', R0_DIR / 'results')
else:
    run_stage0('rollouts-status', '--config', R0_PLAN)


## 26. Restore optional answer-scoring plan (no model loaded)
Requires completed R0, its reconciled reservation, and CPU section32. Scoring must use the sampled production runtime/session. This stage measures the untuned model, with no adapter. A changed runtime is rejected rather than silently joining incompatible evidence.

Scoring records the full output-head vocabulary including padded rows. Rank and NLL use that same vector; the report displays the recorded vocabulary.


In [ ]:
from datetime import datetime, timezone
import uuid

DISTANCE_NAME = STAGE0_NAME + '-answer-distance-v1'
DISTANCE_PLAN = ARTIFACT_ROOT / 'plans' / DISTANCE_NAME / 'stage0.distance.json'
assert DISTANCE_PLAN.is_file(), 'Prepare the scoring plan in CPU section32 first.'
DISTANCE_CONFIG = json.loads(DISTANCE_PLAN.read_text())
DISTANCE_DIR = ARTIFACT_ROOT / DISTANCE_CONFIG['run_path']
DISTANCE_NOTEBOOK = ARTIFACT_ROOT / 'notebook' / DISTANCE_NAME
DISTANCE_NOTEBOOK.mkdir(parents=True, exist_ok=True)
run_stage0('distance-runtime', '--config', DISTANCE_PLAN)

def distance_usage(total, note):
    path = DISTANCE_NOTEBOOK / ('usage-' + uuid.uuid4().hex + '.json')
    observation = dict(event_id=path.stem, session_id=DISTANCE_CONFIG['runtime']['session_id'],
                       total_units=float(total), observed_at=datetime.now(timezone.utc).isoformat(), note=note)
    path.write_text(json.dumps(observation, indent=2) + '\n')
    run_stage0('budget-record', '--config', PLAN, '--observation', path)

def distance_allocation(rate, seconds, notes):
    return dict(reviewer='Prasad', notes=notes, session_id=DISTANCE_CONFIG['runtime']['session_id'],
                source_reviewed=True, max_units_per_hour=float(rate), max_seconds=float(seconds))


## 27. Measure one unique answer (GPU)
Approve a bounded allocation from actual remaining cumulative Stage0 credits. Exactly one missing answer is scored; repeated cohort memberships do not create repeated forwards. To obtain a broader profile, repeat one bounded unit within the existing allocation. No unbounded production is allowed before section29's measured freeze.


In [ ]:
DISTANCE_APPROVE_PROFILE = False
DISTANCE_SPENT_BEFORE_PROFILE = None
DISTANCE_PROFILE_RATE = None
DISTANCE_PROFILE_SECONDS = None
DISTANCE_PROFILE_NOTES = ''
arguments = ['--config', DISTANCE_PLAN, '--max-new-units', 1]
if DISTANCE_APPROVE_PROFILE:
    if (DISTANCE_SPENT_BEFORE_PROFILE is None or DISTANCE_PROFILE_RATE is None
            or DISTANCE_PROFILE_SECONDS is None or not DISTANCE_PROFILE_NOTES.strip()):
        raise ValueError('Review scoring sources and enter cumulative usage, conservative rate/time bounds and notes.')
    distance_usage(DISTANCE_SPENT_BEFORE_PROFILE, 'Cumulative Stage0 credits before bounded answer scoring')
    approval = DISTANCE_NOTEBOOK / ('profile-budget-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(distance_allocation(DISTANCE_PROFILE_RATE, DISTANCE_PROFILE_SECONDS, DISTANCE_PROFILE_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
run_stage0('distance-run', *arguments)
run_stage0('distance-status', '--config', DISTANCE_PLAN)


## 28. Inspect measured forecast and reconcile profile usage (CPU-safe)
The forecast extrapolates the worst measured scoring rate by squared sequence length and includes measured attempt overhead. It is a conservative planning heuristic, not guaranteed completion. Record actual cumulative usage and settle the profile reservation before production approval.


In [ ]:
run_stage0('distance-forecast', '--config', DISTANCE_PLAN)
print((DISTANCE_DIR / 'results' / 'forecast.json').read_text())
DISTANCE_SPENT_AFTER_PROFILE = None
if DISTANCE_SPENT_AFTER_PROFILE is None:
    raise ValueError('Enter the actual cumulative Stage0 compute-unit reading.')
reservation = json.loads((DISTANCE_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
distance_usage(DISTANCE_SPENT_AFTER_PROFILE, 'Cumulative Stage0 credits after bounded scoring profile')
run_stage0('budget-settle', '--reservation-id', reservation)


## 29. Freeze measured scoring production (CPU-safe)
A separate review binds the saved forecast, remaining unique answers and sampled runtime. It must fit the same cumulative 20-unit ceiling. No reset or transfer of another workload's reservation occurs.


In [ ]:
DISTANCE_PRODUCTION_REVIEWED = False
DISTANCE_PRODUCTION_RATE = None
DISTANCE_PRODUCTION_SECONDS = None
DISTANCE_PRODUCTION_NOTES = ''
if (not DISTANCE_PRODUCTION_REVIEWED or DISTANCE_PRODUCTION_RATE is None
        or DISTANCE_PRODUCTION_SECONDS is None or not DISTANCE_PRODUCTION_NOTES.strip()):
    raise ValueError('Review the measured forecast and enter conservative production rate/time limits and notes.')
review = distance_allocation(DISTANCE_PRODUCTION_RATE, DISTANCE_PRODUCTION_SECONDS, DISTANCE_PRODUCTION_NOTES)
review['forecast_sha256'] = __import__('hashlib').sha256((DISTANCE_DIR / 'results' / 'forecast.json').read_bytes()).hexdigest()
review_path = DISTANCE_NOTEBOOK / 'production-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('distance-freeze', '--config', DISTANCE_PLAN, '--review', review_path)


## 30. Score or resume remaining unique answers (GPU)
Set the limit to `None` to finish under the frozen reservation. Sealed token arrays are verified and skipped. Renew only after settling an earlier allocation; changed session/runtime cannot reuse this sampled comparison silently. GPU work ends at measurement completion; bootstraps/report run on CPU.


In [ ]:
DISTANCE_MAX_NEW_ANSWERS = None
DISTANCE_RENEW_BUDGET = False
DISTANCE_SPENT_BEFORE_RENEWAL = None
DISTANCE_RENEW_RATE = None
DISTANCE_RENEW_SECONDS = None
DISTANCE_RENEW_NOTES = ''
arguments = ['--config', DISTANCE_PLAN]
if DISTANCE_RENEW_BUDGET:
    if (DISTANCE_SPENT_BEFORE_RENEWAL is None or DISTANCE_RENEW_RATE is None
            or DISTANCE_RENEW_SECONDS is None or not DISTANCE_RENEW_NOTES.strip()):
        raise ValueError('Settle the old allocation and enter actual usage with new conservative bounds.')
    distance_usage(DISTANCE_SPENT_BEFORE_RENEWAL, 'Cumulative Stage0 credits before scoring renewal')
    approval = DISTANCE_NOTEBOOK / ('renewal-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(distance_allocation(DISTANCE_RENEW_RATE, DISTANCE_RENEW_SECONDS, DISTANCE_RENEW_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if DISTANCE_MAX_NEW_ANSWERS is not None:
    arguments += ['--max-new-units', DISTANCE_MAX_NEW_ANSWERS]
run_stage0('distance-run', *arguments)
run_stage0('distance-status', '--config', DISTANCE_PLAN)


## 31. Reconcile scoring usage and hand off to CPU
Record actual cumulative usage after completion or interruption. Completion verification is CPU-only and never claims a real run from synthetic fixtures. Use CPU section33 for report aggregation and intervals.


In [ ]:
DISTANCE_SPENT_AFTER_COLLECTION = None
if DISTANCE_SPENT_AFTER_COLLECTION is None:
    raise ValueError('Enter actual cumulative Stage0 usage after scoring.')
reservation = json.loads((DISTANCE_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
distance_usage(DISTANCE_SPENT_AFTER_COLLECTION, 'Cumulative Stage0 credits after scoring collection')
run_stage0('budget-settle', '--reservation-id', reservation)
if (DISTANCE_DIR / 'measurement.complete.json').exists():
    run_stage0('distance-verify-measurement', '--config', DISTANCE_PLAN)
    print('GPU measurement saved. Continue on CPU at section33.')
else:
    run_stage0('distance-status', '--config', DISTANCE_PLAN)


## 32. Restore optional initial-gradient plan (no model loaded)
Run setup1–4 and CPU section34 first. A new comparison can start on any suitable pinned FP32 GPU session. The first bounded profile freezes that session; later gradient batches cannot silently switch it. R0 tokens remain source data. This optional branch has its own allocation within the cumulative20-unit ceiling.


In [ ]:
from datetime import datetime, timezone
import uuid
from pilot_eval.stage0_push_backend import PushDependencies

PUSH_NAME = STAGE0_NAME + '-initial-pushes-v1'
PUSH_PLAN = ARTIFACT_ROOT / 'plans' / PUSH_NAME / 'stage0.pushes.json'
assert PUSH_PLAN.is_file(), 'Prepare the comparison in CPU section34 first.'
PUSH_CONFIG = json.loads(PUSH_PLAN.read_text())
PUSH_DIR = ARTIFACT_ROOT / PUSH_CONFIG['run_path']
PUSH_NOTEBOOK = ARTIFACT_ROOT / 'notebook' / PUSH_NAME
PUSH_NOTEBOOK.mkdir(parents=True, exist_ok=True)
PUSH_RUNTIME = PushDependencies().runtime()
run_stage0('pushes-runtime', '--config', PUSH_PLAN)
if not (PUSH_DIR / 'checks' / 'instrument.json').exists():
    run_stage0('pushes-check', '--config', PUSH_PLAN)

def push_usage(total, note):
    path = PUSH_NOTEBOOK / ('usage-' + uuid.uuid4().hex + '.json')
    value = dict(event_id=path.stem, session_id=PUSH_RUNTIME['session_id'],
                 total_units=float(total), observed_at=datetime.now(timezone.utc).isoformat(), note=note)
    path.write_text(json.dumps(value, indent=2) + '\n')
    run_stage0('budget-record', '--config', PLAN, '--observation', path)

def push_allocation(rate, seconds, notes):
    return dict(reviewer='Prasad', notes=notes, session_id=PUSH_RUNTIME['session_id'],
                source_reviewed=True, max_units_per_hour=float(rate), max_seconds=float(seconds))

## 33. Profile one batch per A₀/objective (GPU)
Before production freeze, the runner can collect only one bounded full batch per condition: three or six batches. It loads verified zero adapters, computes masked CE or initial DAPO gradients with a global token denominator, saves raw gradients/pushes and checks G·A₀ reconstruction and exact zero A-gradients. It creates no optimizer. Sealed profile batches are reused later.


In [ ]:
PUSH_PROFILE_REVIEWED = False
PUSH_SPENT_BEFORE_PROFILE = None
PUSH_PROFILE_RATE = None
PUSH_PROFILE_SECONDS = None
PUSH_PROFILE_NOTES = ''
if (not PUSH_PROFILE_REVIEWED or PUSH_SPENT_BEFORE_PROFILE is None
        or PUSH_PROFILE_RATE is None or PUSH_PROFILE_SECONDS is None
        or not PUSH_PROFILE_NOTES.strip()):
    raise ValueError('Review sources and enter actual cumulative usage plus bounded profile rate/time and notes.')
push_usage(PUSH_SPENT_BEFORE_PROFILE, 'Cumulative Stage0 credits before initial-push profile')
approval = PUSH_NOTEBOOK / ('profile-budget-' + uuid.uuid4().hex + '.json')
approval.write_text(json.dumps(push_allocation(PUSH_PROFILE_RATE, PUSH_PROFILE_SECONDS,
                                              PUSH_PROFILE_NOTES), indent=2) + '\n')
run_stage0('pushes-run', '--config', PUSH_PLAN, '--approval', approval)

## 34. Forecast remaining gradients and settle profile usage (CPU-safe)
Inspect condition-specific measured cost extrapolation and remaining batch counts. The forecast includes measured attempt overhead and is a planning estimate. Enter the actual cumulative usage and settle the bounded profile before reserving production.


In [ ]:
run_stage0('pushes-forecast', '--config', PUSH_PLAN)
print((PUSH_DIR / 'results' / 'forecast.json').read_text())
PUSH_SPENT_AFTER_PROFILE = None
if PUSH_SPENT_AFTER_PROFILE is None:
    raise ValueError('Enter the actual cumulative Stage0 compute-unit reading.')
reservation = json.loads((PUSH_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
push_usage(PUSH_SPENT_AFTER_PROFILE, 'Cumulative Stage0 credits after initial-push profile')
run_stage0('budget-settle', '--reservation-id', reservation)

## 35. Freeze measured gradient production (CPU-safe)
A separate review binds the measured forecast and remaining work to the frozen gradient session. This allocation shares the existing cumulative20-unit ceiling; another workload’s approval is not reused.


In [ ]:
PUSH_PRODUCTION_REVIEWED = False
PUSH_PRODUCTION_RATE = None
PUSH_PRODUCTION_SECONDS = None
PUSH_PRODUCTION_NOTES = ''
if (not PUSH_PRODUCTION_REVIEWED or PUSH_PRODUCTION_RATE is None
        or PUSH_PRODUCTION_SECONDS is None or not PUSH_PRODUCTION_NOTES.strip()):
    raise ValueError('Review the forecast; enter conservative production rate/time and notes.')
review = push_allocation(PUSH_PRODUCTION_RATE, PUSH_PRODUCTION_SECONDS, PUSH_PRODUCTION_NOTES)
review['forecast_sha256'] = __import__('hashlib').sha256((PUSH_DIR / 'results' / 'forecast.json').read_bytes()).hexdigest()
review_path = PUSH_NOTEBOOK / 'production-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('pushes-freeze', '--config', PUSH_PLAN, '--review', review_path)

## 36. Collect or resume initial gradients (GPU)
Only missing sealed question-batches are measured. All objectives stay within the frozen gradient session. GPU work ends when measurement completes; summaries and endpoint reveal run on CPU. A failed gate preserves evidence and stops without widening tolerances.


In [ ]:
PUSH_MAX_NEW_BATCHES = None
PUSH_RENEW_BUDGET = False
PUSH_SPENT_BEFORE_RENEWAL = None
PUSH_RENEW_RATE = None
PUSH_RENEW_SECONDS = None
PUSH_RENEW_NOTES = ''
arguments = ['--config', PUSH_PLAN]
if PUSH_RENEW_BUDGET:
    if (PUSH_SPENT_BEFORE_RENEWAL is None or PUSH_RENEW_RATE is None
            or PUSH_RENEW_SECONDS is None or not PUSH_RENEW_NOTES.strip()):
        raise ValueError('Enter actual usage and reviewed renewal bounds.')
    push_usage(PUSH_SPENT_BEFORE_RENEWAL, 'Cumulative Stage0 credits before initial-push renewal')
    reservation = json.loads((PUSH_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
    run_stage0('budget-settle', '--reservation-id', reservation)
    approval = PUSH_NOTEBOOK / ('renewal-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(push_allocation(PUSH_RENEW_RATE, PUSH_RENEW_SECONDS,
                                                  PUSH_RENEW_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if PUSH_MAX_NEW_BATCHES is not None:
    arguments += ['--max-new-units', PUSH_MAX_NEW_BATCHES]
run_stage0('pushes-run', *arguments)

## 37. Reconcile gradient usage and hand off to CPU
Record actual cumulative usage even after interruption. Completion verifies raw gradient seals and mathematical gates. Continue with CPU sections35–36 for summaries, conditional predictions and endpoint reveal; no GPU is needed there.


In [ ]:
PUSH_SPENT_AFTER_COLLECTION = None
if PUSH_SPENT_AFTER_COLLECTION is None:
    raise ValueError('Enter actual cumulative Stage0 usage after gradient collection.')
reservation = json.loads((PUSH_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
push_usage(PUSH_SPENT_AFTER_COLLECTION, 'Cumulative Stage0 credits after initial-push collection')
run_stage0('budget-settle', '--reservation-id', reservation)
if (PUSH_DIR / 'measurement.complete.json').exists():
    run_stage0('pushes-verify-measurement', '--config', PUSH_PLAN)
    print('Gradient measurement saved. Continue on CPU at35–36.')
else:
    run_stage0('pushes-status', '--config', PUSH_PLAN)

## 38. Restore optional base-input energy plan (no model loaded)
Run setup1–4 and CPU37–38 first. The first bounded energy profile freezes its own GPU/runtime. Later groups stay in that session. Qualification is a source/instrument prerequisite; its historical numerical outputs are not joined to new energy values. All work shares the existing20-unit budget.


In [ ]:
from datetime import datetime, timezone
import uuid
from pilot_eval.stage0_energy_backend import EnergyDependencies

ENERGY_NAME = STAGE0_NAME + '-input-energy-v1'
ENERGY_PLAN = ARTIFACT_ROOT / 'plans' / ENERGY_NAME / 'stage0.energy.json'
assert ENERGY_PLAN.is_file(), 'Prepare the energy plan in CPU37 and freeze predictions in38 first.'
ENERGY_CONFIG = json.loads(ENERGY_PLAN.read_text())
ENERGY_DIR = ARTIFACT_ROOT / ENERGY_CONFIG['run_path']
ENERGY_NOTEBOOK = ARTIFACT_ROOT / 'notebook' / ENERGY_NAME
ENERGY_NOTEBOOK.mkdir(parents=True, exist_ok=True)
ENERGY_RUNTIME = EnergyDependencies().runtime()
run_stage0('energy-runtime', '--config', ENERGY_PLAN)
assert (ENERGY_DIR / 'predictions.complete.json').is_file(), 'Freeze energy hypotheses in CPU38 first.'

def energy_usage(total, note):
    path = ENERGY_NOTEBOOK / ('usage-' + uuid.uuid4().hex + '.json')
    value = dict(event_id=path.stem, session_id=ENERGY_RUNTIME['session_id'],
                 total_units=float(total), observed_at=datetime.now(timezone.utc).isoformat(), note=note)
    path.write_text(json.dumps(value, indent=2) + '\n')
    run_stage0('budget-record', '--config', PLAN, '--observation', path)

def energy_allocation(rate, seconds, notes):
    return dict(reviewer='Prasad', notes=notes, session_id=ENERGY_RUNTIME['session_id'],
                source_reviewed=True, max_units_per_hour=float(rate), max_seconds=float(seconds))

## 39. Profile base-input capture (GPU)
The unfrozen plan permits only its first GSM8K and FineWeb groups:16 documents at the default save grouping. Each sequence gets one untuned body forward, supplying all five factor conditions and three cutoffs. Inputs stay FP32; contractions and sums are FP64. Total energy comes independently from a·x. Profile arrays are reused in production.


In [ ]:
if (ENERGY_DIR / 'production-freeze.json').exists():
    raise RuntimeError('Production is already frozen; resume in section42 rather than reserving another profile.')
ENERGY_PROFILE_REVIEWED = False
ENERGY_SPENT_BEFORE_PROFILE = None
ENERGY_PROFILE_RATE = None
ENERGY_PROFILE_SECONDS = None
ENERGY_PROFILE_NOTES = ''
if (not ENERGY_PROFILE_REVIEWED or ENERGY_SPENT_BEFORE_PROFILE is None
        or ENERGY_PROFILE_RATE is None or ENERGY_PROFILE_SECONDS is None
        or not ENERGY_PROFILE_NOTES.strip()):
    raise ValueError('Review sources and enter actual cumulative usage plus bounded profile rate/time and notes.')
energy_usage(ENERGY_SPENT_BEFORE_PROFILE, 'Cumulative Stage0 credits before input-energy profile')
approval = ENERGY_NOTEBOOK / ('profile-budget-' + uuid.uuid4().hex + '.json')
approval.write_text(json.dumps(energy_allocation(ENERGY_PROFILE_RATE, ENERGY_PROFILE_SECONDS,
                                              ENERGY_PROFILE_NOTES), indent=2) + '\n')
run_stage0('energy-run', '--config', ENERGY_PLAN, '--approval', approval)

## 40. Inspect energy numerics, forecast and settle profile (CPU-safe)
Inspect measured cost, exact reconstruction errors/allowance fractions, normalized projection checks and remaining base-forward count. The time forecast is a planning estimate. Record actual cumulative usage and settle the profile before production.


In [ ]:
run_stage0('energy-forecast', '--config', ENERGY_PLAN)
print((ENERGY_DIR / 'results' / 'forecast.json').read_text())
ENERGY_SPENT_AFTER_PROFILE = None
if ENERGY_SPENT_AFTER_PROFILE is None:
    raise ValueError('Enter the actual cumulative Stage0 compute-unit reading.')
reservation = json.loads((ENERGY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
energy_usage(ENERGY_SPENT_AFTER_PROFILE, 'Cumulative Stage0 credits after input-energy profile')
run_stage0('budget-settle', '--reservation-id', reservation)

## 41. Freeze measured energy production (CPU-safe)
Review the saved numerical evidence and measured forecast, then enter a separately bounded production allocation. Gates remain atol1e-5/rtol1e-5; no fitted tolerance. A new workload cannot spend another reservation. The cumulative ceiling stays20 units.


In [ ]:
ENERGY_PRODUCTION_REVIEWED = False
ENERGY_PRODUCTION_RATE = None
ENERGY_PRODUCTION_SECONDS = None
ENERGY_PRODUCTION_NOTES = ''
if (not ENERGY_PRODUCTION_REVIEWED or ENERGY_PRODUCTION_RATE is None
        or ENERGY_PRODUCTION_SECONDS is None or not ENERGY_PRODUCTION_NOTES.strip()):
    raise ValueError('Review the forecast; enter conservative production rate/time and notes.')
review = energy_allocation(ENERGY_PRODUCTION_RATE, ENERGY_PRODUCTION_SECONDS, ENERGY_PRODUCTION_NOTES)
review.update(numerical_reviewed=True, agreement=ENERGY_CONFIG['agreement'])
review['forecast_sha256'] = __import__('hashlib').sha256((ENERGY_DIR / 'results' / 'forecast.json').read_bytes()).hexdigest()
review_path = ENERGY_NOTEBOOK / 'production-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('energy-freeze', '--config', ENERGY_PLAN, '--review', review_path)

## 42. Collect or resume base-input energy sums (GPU)
Measures only missing save groups. All factors share the untuned inputs; no tuned-model forward, output-head projection, backward pass or optimizer is needed. Sealed groups are skipped. Failed reconstruction preserves diagnostics and stops. GPU work ends at measurement completion.


In [ ]:
ENERGY_MAX_NEW_BATCHES = None
ENERGY_RENEW_BUDGET = False
ENERGY_SPENT_BEFORE_RENEWAL = None
ENERGY_RENEW_RATE = None
ENERGY_RENEW_SECONDS = None
ENERGY_RENEW_NOTES = ''
arguments = ['--config', ENERGY_PLAN]
if ENERGY_RENEW_BUDGET:
    if (ENERGY_SPENT_BEFORE_RENEWAL is None or ENERGY_RENEW_RATE is None
            or ENERGY_RENEW_SECONDS is None or not ENERGY_RENEW_NOTES.strip()):
        raise ValueError('Enter actual usage and reviewed renewal bounds.')
    energy_usage(ENERGY_SPENT_BEFORE_RENEWAL, 'Cumulative Stage0 credits before input-energy renewal')
    reservation = json.loads((ENERGY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
    run_stage0('budget-settle', '--reservation-id', reservation)
    approval = ENERGY_NOTEBOOK / ('renewal-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(energy_allocation(ENERGY_RENEW_RATE, ENERGY_RENEW_SECONDS,
                                                  ENERGY_RENEW_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if ENERGY_MAX_NEW_BATCHES is not None:
    arguments += ['--max-new-units', ENERGY_MAX_NEW_BATCHES]
run_stage0('energy-run', *arguments)

## 43. Reconcile energy usage and hand off to CPU
Record actual cumulative usage after completion or interruption. Verify saved measurement receipts, then use CPU39 for aggregation and plots. Interruption preserves sealed groups; a changed GPU/session cannot silently continue the same comparison.


In [ ]:
ENERGY_SPENT_AFTER_COLLECTION = None
if ENERGY_SPENT_AFTER_COLLECTION is None:
    raise ValueError('Enter actual cumulative Stage0 usage after energy collection.')
reservation = json.loads((ENERGY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
energy_usage(ENERGY_SPENT_AFTER_COLLECTION, 'Cumulative Stage0 credits after input-energy collection')
run_stage0('budget-settle', '--reservation-id', reservation)
if (ENERGY_DIR / 'measurement.complete.json').exists():
    run_stage0('energy-verify-measurement', '--config', ENERGY_PLAN)
    print('Gradient measurement saved. Continue on CPU at39.')
else:
    run_stage0('energy-status', '--config', ENERGY_PLAN)

## 44. Prepare missing-only checkpoint measurement (no model loaded)
After setup1–4 and CPU40–41, inspect the current GPU runtime and freeze a separately named missing-measurement plan. Reuse requires exact recorded session/input/scheduling identities and verified paired KL/write records. A different GPU session recollects the paired values under its own group rather than joining historical measurements. Select only available original aliases. Preparation does not reserve credits or load weights.


In [ ]:
from datetime import datetime, timezone
import uuid
from pilot_eval.stage0_drift_backend import DriftDependencies

TRAJECTORY_NAME = STAGE0_NAME + '-checkpoint-trajectories-v1'
TRAJECTORY_PLAN = ARTIFACT_ROOT / 'plans' / TRAJECTORY_NAME / 'stage0.trajectory.json'
TRAJECTORY_COLLECTION_NAME = TRAJECTORY_NAME + '-collection-v1'
TRAJECTORY_COLLECTION_PLAN = ARTIFACT_ROOT / 'plans' / TRAJECTORY_COLLECTION_NAME / 'stage0.trajectory.collection.json'
TRAJECTORY_QUALIFICATION = ARTIFACT_ROOT / 'plans' / (STAGE0_NAME + '-loader-qualification-v1') / 'stage0.qualification.json'
TRAJECTORY_STATES = None  # None selects all verified original states; or e.g. ['sft-64', 'grpo-64'].
run_stage0('trajectory-runtime', '--config', TRAJECTORY_PLAN)
arguments = ['--config', TRAJECTORY_PLAN, '--qualification', TRAJECTORY_QUALIFICATION,
             '--name', TRAJECTORY_COLLECTION_NAME, '--context-chunk', 32,
             '--workspace-bytes', 512 * 1024 * 1024]
if TRAJECTORY_STATES is not None:
    arguments += ['--states', *TRAJECTORY_STATES]
run_stage0('trajectory-collection-prepare', *arguments)
TRAJECTORY_CONFIG = json.loads(TRAJECTORY_COLLECTION_PLAN.read_text())
TRAJECTORY_DIR = ARTIFACT_ROOT / TRAJECTORY_CONFIG['run_path']
TRAJECTORY_NOTEBOOK = ARTIFACT_ROOT / 'notebook' / TRAJECTORY_COLLECTION_NAME
TRAJECTORY_NOTEBOOK.mkdir(parents=True, exist_ok=True)
TRAJECTORY_RUNTIME = DriftDependencies().runtime()
print((TRAJECTORY_DIR / 'meta' / 'run_manifest.json').read_text())

def trajectory_usage(total, note):
    path = TRAJECTORY_NOTEBOOK / ('usage-' + uuid.uuid4().hex + '.json')
    value = dict(event_id=path.stem, session_id=TRAJECTORY_RUNTIME['session_id'], total_units=float(total),
                 observed_at=datetime.now(timezone.utc).isoformat(), note=note)
    path.write_text(json.dumps(value, indent=2) + '\n')
    run_stage0('budget-record', '--config', PLAN, '--observation', path)

def trajectory_allocation(rate, seconds, notes):
    return dict(reviewer='Prasad', notes=notes, session_id=TRAJECTORY_RUNTIME['session_id'],
                source_reviewed=True, max_units_per_hour=float(rate), max_seconds=float(seconds))

## 45. Freeze hypotheses and profile only missing states (GPU)
Write hypotheses and declare prior exposure. A bounded first two-document group per missing checkpoint/corpus measures costs using Ticket06’s strict repeat/chunk/rank1/KL/reference gates. Profile units are reused. New KL and effective writes are collected together; raw coefficient production is not added. If everything was reusable, run once to seal completion and skip46–48.


In [ ]:
TRAJECTORY_PREDICTIONS = ''
TRAJECTORY_PRIOR_EXPOSURE = None
TRAJECTORY_PREDICTION_NOTES = ''
if (not TRAJECTORY_PREDICTIONS.strip() or type(TRAJECTORY_PRIOR_EXPOSURE) is not bool
        or not TRAJECTORY_PREDICTION_NOTES.strip()):
    raise ValueError('Enter supporting hypotheses, prior exposure and notes.')
review = dict(reviewer='Prasad', notes=TRAJECTORY_PREDICTION_NOTES,
              predictions=TRAJECTORY_PREDICTIONS, prior_exposure=TRAJECTORY_PRIOR_EXPOSURE)
review_path = TRAJECTORY_NOTEBOOK / 'predictions-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('trajectory-collection-predict', '--config', TRAJECTORY_COLLECTION_PLAN, '--review', review_path)
if not TRAJECTORY_CONFIG['units']:
    run_stage0('trajectory-collection-run', '--config', TRAJECTORY_COLLECTION_PLAN)
    print('All selected values reused; no model loaded or allocation needed. Continue with CPU42.')
else:
    if (TRAJECTORY_DIR / 'production-freeze.json').exists():
        raise RuntimeError('Production is already frozen; resume in48 instead of reserving another profile.')
    TRAJECTORY_PROFILE_REVIEWED = False
    TRAJECTORY_SPENT_BEFORE_PROFILE = None
    TRAJECTORY_PROFILE_RATE = None
    TRAJECTORY_PROFILE_SECONDS = None
    TRAJECTORY_PROFILE_NOTES = ''
    if (not TRAJECTORY_PROFILE_REVIEWED or TRAJECTORY_SPENT_BEFORE_PROFILE is None
            or TRAJECTORY_PROFILE_RATE is None or TRAJECTORY_PROFILE_SECONDS is None
            or not TRAJECTORY_PROFILE_NOTES.strip()):
        raise ValueError('Review missing work and enter actual cumulative usage plus conservative profile bounds.')
    trajectory_usage(TRAJECTORY_SPENT_BEFORE_PROFILE, 'Cumulative Stage0 credits before trajectory profile')
    approval = TRAJECTORY_NOTEBOOK / ('profile-budget-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(trajectory_allocation(TRAJECTORY_PROFILE_RATE, TRAJECTORY_PROFILE_SECONDS,
                                                        TRAJECTORY_PROFILE_NOTES), indent=2) + '\n')
    run_stage0('trajectory-collection-run', '--config', TRAJECTORY_COLLECTION_PLAN, '--approval', approval)

## 46. Inspect measured forecast and reconcile profile (CPU-safe)
The forecast extrapolates each checkpoint/corpus from measured timing, plus whole-attempt overhead. Inspect missing adapter-forward counts and extra reference/strict diagnostic calls; it is not a completion guarantee. Enter actual cumulative usage and settle the profile before production. Skip if45 reused everything.


In [ ]:
run_stage0('trajectory-collection-forecast', '--config', TRAJECTORY_COLLECTION_PLAN)
print((TRAJECTORY_DIR / 'results' / 'forecast.json').read_text())
TRAJECTORY_SPENT_AFTER_PROFILE = None
if TRAJECTORY_SPENT_AFTER_PROFILE is None:
    raise ValueError('Enter the actual cumulative Stage0 credit reading.')
reservation = json.loads((TRAJECTORY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
trajectory_usage(TRAJECTORY_SPENT_AFTER_PROFILE, 'Cumulative Stage0 credits after trajectory profile')
run_stage0('budget-settle', '--reservation-id', reservation)

## 47. Freeze separately bounded trajectory production (CPU-safe)
Bind the measured forecast, frozen session and remaining missing work to a separate production review. The same cumulative20-unit ceiling applies across all Stage0 tickets. No interpolation/equal-size experiment or new training is authorized here.


In [ ]:
TRAJECTORY_PRODUCTION_REVIEWED = False
TRAJECTORY_PRODUCTION_RATE = None
TRAJECTORY_PRODUCTION_SECONDS = None
TRAJECTORY_PRODUCTION_NOTES = ''
if (not TRAJECTORY_PRODUCTION_REVIEWED or TRAJECTORY_PRODUCTION_RATE is None
        or TRAJECTORY_PRODUCTION_SECONDS is None or not TRAJECTORY_PRODUCTION_NOTES.strip()):
    raise ValueError('Review the measured forecast and enter conservative production bounds.')
review = trajectory_allocation(TRAJECTORY_PRODUCTION_RATE, TRAJECTORY_PRODUCTION_SECONDS, TRAJECTORY_PRODUCTION_NOTES)
review['forecast_sha256'] = __import__('hashlib').sha256((TRAJECTORY_DIR / 'results' / 'forecast.json').read_bytes()).hexdigest()
review_path = TRAJECTORY_NOTEBOOK / 'production-review.json'
review_path.write_text(json.dumps(review, indent=2) + '\n')
run_stage0('trajectory-collection-freeze', '--config', TRAJECTORY_COLLECTION_PLAN, '--review', review_path)

## 48. Measure or resume remaining checkpoint KL/write pairs (GPU)
Only missing sealed groups run. KL is full-vocabulary variant||untuned in nats; writes use the exact Ticket06 per-module norm ratio on adapted inputs, with unshifted positions. No raw-coefficient acceptance policy is widened. Changed runtime/source or strict gate failure stops. All original checkpoints share this recorded run.


In [ ]:
TRAJECTORY_MAX_NEW_BATCHES = None
TRAJECTORY_RENEW_BUDGET = False
TRAJECTORY_SPENT_BEFORE_RENEWAL = None
TRAJECTORY_RENEW_RATE = None
TRAJECTORY_RENEW_SECONDS = None
TRAJECTORY_RENEW_NOTES = ''
arguments = ['--config', TRAJECTORY_COLLECTION_PLAN]
if TRAJECTORY_RENEW_BUDGET:
    if (TRAJECTORY_SPENT_BEFORE_RENEWAL is None or TRAJECTORY_RENEW_RATE is None
            or TRAJECTORY_RENEW_SECONDS is None or not TRAJECTORY_RENEW_NOTES.strip()):
        raise ValueError('Enter actual usage and reviewed renewal bounds.')
    trajectory_usage(TRAJECTORY_SPENT_BEFORE_RENEWAL, 'Cumulative Stage0 credits before trajectory renewal')
    reservation = json.loads((TRAJECTORY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
    run_stage0('budget-settle', '--reservation-id', reservation)
    approval = TRAJECTORY_NOTEBOOK / ('renewal-' + uuid.uuid4().hex + '.json')
    approval.write_text(json.dumps(trajectory_allocation(TRAJECTORY_RENEW_RATE, TRAJECTORY_RENEW_SECONDS,
                                                  TRAJECTORY_RENEW_NOTES), indent=2) + '\n')
    arguments += ['--approval', approval]
if TRAJECTORY_MAX_NEW_BATCHES is not None:
    arguments += ['--max-new-units', TRAJECTORY_MAX_NEW_BATCHES]
run_stage0('trajectory-collection-run', *arguments)

## 49. Reconcile trajectory usage and verify handoff
Record actual cumulative credits after completion or interruption. CPU verification checks per-position arrays, mappings, checkpoint hashes and sufficient-sum reconstruction. Then CPU42 can include this collection in a new report snapshot. No generation or equal-size rescaling runs here.


In [ ]:
TRAJECTORY_SPENT_AFTER_COLLECTION = None
if TRAJECTORY_SPENT_AFTER_COLLECTION is None:
    raise ValueError('Enter actual cumulative Stage0 usage after trajectory collection.')
reservation = json.loads((TRAJECTORY_DIR / 'meta' / 'active-reservation.json').read_text())['reservation_id']
trajectory_usage(TRAJECTORY_SPENT_AFTER_COLLECTION, 'Cumulative Stage0 credits after trajectory collection')
run_stage0('budget-settle', '--reservation-id', reservation)
if (TRAJECTORY_DIR / 'complete.json').exists():
    run_stage0('trajectory-collection-verify', '--config', TRAJECTORY_COLLECTION_PLAN)
    print('Saved. CPU42 can now set INCLUDE_TRAJECTORY_COLLECTION=True and select a new report name.')
else:
    run_stage0('trajectory-collection-status', '--config', TRAJECTORY_COLLECTION_PLAN)